# RoBERTa Fine-Tuning for Adversarial Prompt Detection

Fine-tunes `roberta-base` as a binary classifier (adversarial vs benign) on pre-built splits.

**Prerequisites:**
- Upload `train.parquet`, `val.parquet`, `test.parquet`, `test_unseen.parquet` from `data/processed/splits/` to your Google Drive
- Select a T4 GPU runtime

## 1. Setup & Environment

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow matplotlib seaborn

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/llm_gatekeeping"
DATA_DIR = os.path.join(DRIVE_ROOT, "splits")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "roberta_classifier")
PREDICTIONS_DIR = os.path.join(DRIVE_ROOT, "predictions")

# Cache HF downloads on Drive so they persist across sessions
os.environ["HF_HOME"] = os.path.join(DRIVE_ROOT, ".hf_cache")

for d in [DATA_DIR, OUTPUT_DIR, PREDICTIONS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Data dir:        {DATA_DIR}")
print(f"Output dir:      {OUTPUT_DIR}")
print(f"Predictions dir: {PREDICTIONS_DIR}")

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Data Loading

In [ ]:
SPLIT_NAMES = ["train", "val", "test", "test_unseen"]

GROUND_TRUTH_COLS = [
    "modified_sample",
    "original_sample",
    "attack_name",
    "label_binary",
    "label_category",
    "label_type",
    "prompt_hash",
]

LABEL2ID = {"benign": 0, "adversarial": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

splits = {}
for name in SPLIT_NAMES:
    path = os.path.join(DATA_DIR, f"{name}.parquet")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Upload data/processed/splits/{name}.parquet to {DATA_DIR}"
        )
    splits[name] = pd.read_parquet(path)
    print(f"{name}: {len(splits[name]):,} samples")

In [ ]:
for name, df in splits.items():
    counts = df["label_binary"].value_counts()
    print(f"\n{name}:")
    for label, count in counts.items():
        print(f"  {label}: {count:,} ({count / len(df) * 100:.1f}%)")

## 3. Tokenization

In [ ]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def make_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict(
        {
            "text": df["modified_sample"].tolist(),
            "label": [LABEL2ID[l] for l in df["label_binary"]],
        }
    )
    ds = ds.map(
        lambda batch: tokenizer(
            batch["text"], truncation=True, max_length=MAX_LENGTH
        ),
        batched=True,
        remove_columns=["text"],
    )
    return ds


datasets = {name: make_dataset(df) for name, df in splits.items()}
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenization complete.")
print(f"Features: {datasets['train'].column_names}")

## 4. Model & Training

In [ ]:
# Compute balanced class weights to handle ~11:1 imbalance
train_labels = splits["train"]["label_binary"].map(LABEL2ID).values
class_weights = compute_class_weight(
    "balanced", classes=np.array([0, 1]), y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights: benign={class_weights[0]:.3f}, adversarial={class_weights[1]:.3f}")

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer with weighted cross-entropy loss for class imbalance."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = torch.nn.functional.cross_entropy(
            logits, labels, weight=class_weights_tensor
        )
        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, pos_label=1),
        "precision": precision_score(labels, preds, pos_label=1),
        "recall": recall_score(labels, preds, pos_label=1),
    }


training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    seed=42,
    report_to="none",
)

print("TrainingArguments configured.")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Model parameters: {model.num_parameters():,}")
trainer.train()

## 5. Training Curves

In [ ]:
log_history = trainer.state.log_history

train_steps = [e["step"] for e in log_history if "loss" in e]
train_loss = [e["loss"] for e in log_history if "loss" in e]
eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
eval_loss = [e["eval_loss"] for e in log_history if "eval_loss" in e]
eval_f1 = [e["eval_f1"] for e in log_history if "eval_f1" in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_steps, train_loss, label="Train", alpha=0.7)
ax1.plot(eval_steps, eval_loss, label="Val", marker="o", markersize=3)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(eval_steps, eval_f1, marker="o", markersize=3, color="green")
ax2.set_xlabel("Step")
ax2.set_ylabel("F1")
ax2.set_title("Validation F1")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 6. Evaluation

In [ ]:
def predict_split(trainer, dataset, df):
    """Run predictions and return a DataFrame with pipeline-compatible columns."""
    output = trainer.predict(dataset)
    logits = output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(logits, axis=-1)

    result = df.copy()
    result["roberta_pred_binary"] = [ID2LABEL[p] for p in preds]
    result["roberta_conf_binary"] = np.max(probs, axis=-1)
    result["roberta_proba_binary_benign"] = probs[:, 0]
    result["roberta_proba_binary_adversarial"] = probs[:, 1]
    return result

In [ ]:
results = {}
all_metrics = {}

for name in ["val", "test", "test_unseen"]:
    result_df = predict_split(trainer, datasets[name], splits[name])
    results[name] = result_df

    y_true = result_df["label_binary"]
    y_pred = result_df["roberta_pred_binary"]

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label="adversarial"),
        "precision": precision_score(y_true, y_pred, pos_label="adversarial"),
        "recall": recall_score(y_true, y_pred, pos_label="adversarial"),
    }

    # ROC-AUC: guard against single-class splits
    unique_labels = y_true.nunique()
    if unique_labels > 1:
        metrics["roc_auc"] = roc_auc_score(
            (y_true == "adversarial").astype(int),
            result_df["roberta_proba_binary_adversarial"],
        )
    else:
        metrics["roc_auc"] = float("nan")

    # Benign recall
    benign_mask = y_true == "benign"
    if benign_mask.any():
        metrics["benign_recall"] = (y_pred[benign_mask] == "benign").mean()
    else:
        metrics["benign_recall"] = float("nan")

    metrics["n_samples"] = len(result_df)
    all_metrics[name] = metrics

    print(f"\n{'=' * 50}")
    print(f"{name.upper()} ({len(result_df):,} samples)")
    print(f"{'=' * 50}")
    for k, v in metrics.items():
        if k != "n_samples":
            print(f"  {k}: {v:.4f}")
    print(f"\n{classification_report(y_true, y_pred)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels_order = ["benign", "adversarial"]

for ax, name in zip(axes, ["val", "test", "test_unseen"]):
    df = results[name]
    cm = confusion_matrix(
        df["label_binary"], df["roberta_pred_binary"], labels=labels_order
    )
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels_order,
        yticklabels=labels_order,
        ax=ax,
    )
    ax.set_title(f"{name}")
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=150)
plt.show()

## 7. Save & Export

In [ ]:
# Save model (float32), tokenizer, label mapping, and metrics
model_dir = os.path.join(OUTPUT_DIR, "model")
tokenizer_dir = os.path.join(OUTPUT_DIR, "tokenizer")

# Save in float32 for portability
model_f32 = model.float()
model_f32.save_pretrained(model_dir)
tokenizer.save_pretrained(tokenizer_dir)

with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w") as f:
    json.dump({"label2id": LABEL2ID, "id2label": ID2LABEL}, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

print(f"Model saved to {model_dir}")
print(f"Tokenizer saved to {tokenizer_dir}")
print(f"Metrics saved to {os.path.join(OUTPUT_DIR, 'metrics.json')}")

In [ ]:
# Export pipeline-compatible prediction parquets
pred_cols = [
    "roberta_pred_binary",
    "roberta_conf_binary",
    "roberta_proba_binary_benign",
    "roberta_proba_binary_adversarial",
]

for name in ["val", "test", "test_unseen"]:
    df = results[name]
    gt_cols = [c for c in GROUND_TRUTH_COLS if c in df.columns]
    export_cols = gt_cols + pred_cols
    if "sample_id" in df.columns:
        export_cols = ["sample_id"] + export_cols

    out_path = os.path.join(PREDICTIONS_DIR, f"roberta_predictions_{name}.parquet")
    df[export_cols].to_parquet(out_path, index=False)
    print(f"Saved {out_path} ({len(df):,} rows)")